In [1]:
import os
import pandas as pd
import numpy as np
import gc

# -------------------------------------
# From combined_data1.csv stratify split by label for detection
# For mitigation stratify split by multi-labels
# ---------------------------------------


# datapath
datapath = 'D:/fypadpm/dataset/CSECICIDS2018/'

## Pre-Processing for Intrusion Detection Dataset

### Stratify Split by Label

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.concat([
    pd.read_csv(datapath+"df_train_setm.csv", low_memory=False),
    pd.read_csv(datapath+"df_validation_setm.csv", low_memory=False)], axis=0, ignore_index=True)


# Check that Attack_Type exists
print(df['Label'].value_counts())

# ==========================
# STRATIFIED SPLIT
# ==========================
df_train, df_temp = train_test_split(
    df,
    test_size=0.3,         # change as needed e.g. 0.2 = 20% validation
    random_state=42,
    stratify=df['Label']
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,         # change as needed e.g. 0.2 = 20% validation
    random_state=42,
    stratify=df_temp['Label']
)

# ==========================
# Save datasets
# ===========================
df_train.to_csv(datapath+"df_train_str.csv", index=False)
df_val.to_csv(datapath+"df_validation_str.csv", index=False)
df_test.to_csv(datapath+"df_test_str.csv", index=False)

print("Done! Stratified train/val split saved.")
print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(df_val)}")
print(f"Test size: {len(df_test)}")

del df, df_train, df_val, df_test, df_temp
gc.collect()

Label
Benign                    11024459
DDOS attack-HOIC            579050
DDoS attacks-LOIC-HTTP      576191
DoS attacks-Hulk            341516
Bot                         236327
Infilteration               144281
SSH-Bruteforce              117322
DoS attacks-GoldenEye        41455
FTP-BruteForce               27338
DoS attacks-Slowloris         8965
Brute Force -Web               480
Brute Force -XSS               230
SQL Injection                   85
Name: count, dtype: int64
Done! Stratified train/val split saved.
Train size: 9168389
Validation size: 1964655
Test size: 1964655


0

### Preprocessing on Train, validation and Test Datasets of Intrusion Detection

In [25]:
import pandas as pd
import numpy as np

def preprocess_train_val_df(df):
    """
    Optimized preprocessing for a single dataframe.
    No df.copy() used. All operations happen in-place.
    """

    # 1️⃣ Drop Date
    if "Date" in df.columns:
        df.drop(columns="Date", inplace=True)

    # 2️⃣ Convert Timestamp
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")

    # 3️⃣ Convert non-excluded to numeric
    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID']
    cols_to_convert = [col for col in df.columns if col not in exclude_cols]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 4️⃣ Remove negative Flow IAT Min/Max
    if "Flow IAT Min" in df.columns:
        df = df[df["Flow IAT Min"] >= 0]
    if "Flow IAT Max" in df.columns:
        df = df[df["Flow IAT Max"] >= 0]

    # 5️⃣ Remove INF rows
    cols_inf = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if cols_inf:
        df = df[~df[cols_inf].isin([np.inf, -np.inf]).any(axis=1)]

    # 6️⃣ Fill string columns
    for col in ["Dst IP", "Src IP", "Flow ID"]:
        if col in df.columns:
            df[col] = df[col].fillna("<<M>>")

    # 7️⃣ Fill numeric columns
    if "Src Port" in df.columns:
        df["Src Port"] = df["Src Port"].fillna(0)

    # 8️⃣ Drop NaN in critical numeric columns
    critical = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if critical:
        df.dropna(subset=critical, inplace=True)

    # 9️⃣ Drop all-zero numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols).tolist()
    zero_cols = [col for col in numeric_cols if (df[col] == 0).all()]
    if zero_cols:
        df.drop(columns=zero_cols, inplace=True)

    # 🔟 Final diagnostics
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols).tolist()
    neg_cols = (df[numeric_cols] < 0).sum()
    inf_cols = np.isinf(df[numeric_cols]).sum()
    null_cols = df.isnull().sum()

    print("Columns with negative values:")
    print(neg_cols[neg_cols > 0])

    print("\nColumns with inf values:")
    print(inf_cols[inf_cols > 0])

    print("\nColumns with null values:")
    print(null_cols[null_cols > 0])

    print("\nFinal shape:", df.shape)

    return df



In [26]:
df_train = pd.read_csv(datapath+"df_train_str.csv", low_memory=False)
df_train = preprocess_train_val_df(df_train)
df_val = pd.read_csv(datapath+"df_validation_str.csv", low_memory=False)
df_val = preprocess_train_val_df(df_val)

df_train.to_csv(datapath+"processed_train_set.csv", index=False)
df_val.to_csv(datapath+"processed_validation_set.csv", index=False)

del df_train, df_val
gc.collect()


C:\Users\loke\AppData\Local\Temp\ipykernel_1596\317745539.py:16: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")


Columns with negative values:
Init Bwd Win Byts    3172640
Init Fwd Win Byts    1729068
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (6178022, 77)


C:\Users\loke\AppData\Local\Temp\ipykernel_1596\317745539.py:16: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")


Columns with negative values:
Init Bwd Win Byts    1003172
Init Fwd Win Byts     546836
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (1952907, 77)


392

In [27]:
import pandas as pd
import numpy as np

def preprocess_test_df(df):
    """
    Preprocess a single DataFrame.
    """
    
    # Drop Date
    if 'Date' in df.columns:
        df = df.drop(columns='Date')
    
    # Convert Timestamp
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')
    
    # Convert columns to numeric
    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID']
    cols_to_convert = [col for col in df.columns if col not in exclude_cols]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Fill missing string/numeric columns
    for col in ['Dst IP', 'Src IP', 'Flow ID']:
        if col in df.columns:
            df[col] = df[col].fillna("<<M>>")
    for col in ['Src Port', 'Dst Port', 'Protocol']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # Drop NaN in critical numeric columns
    critical_numeric = ['Flow Byts/s', 'Flow Pkts/s']
    df = df.dropna(subset=[c for c in critical_numeric if c in df.columns])
    
    # Remove inf/-inf in numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols).tolist()
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    
    # Remove negative values (except allowed columns)
    allowed_negative_cols = ['Init Fwd Win Byts', 'Init Bwd Win Byts']
    cols_to_check_neg = [col for col in numeric_cols if col not in allowed_negative_cols]
    df = df[(df[cols_to_check_neg] >= 0).all(axis=1)]
    
    # Drop all-zero numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols).tolist()
    zero_cols = [col for col in numeric_cols if (df[col] == 0).all()]
    if zero_cols:
        df = df.drop(columns=zero_cols)
    
    # Final validation
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols).tolist()
    neg_counts = (df[numeric_cols] < 0).sum()
    inf_counts = np.isinf(df[numeric_cols]).sum()
    null_counts = df.isna().sum()
    
    print("Numeric columns with negative values:\n", neg_counts[neg_counts > 0])
    print("\nNumeric columns with infinite values:\n", inf_counts[inf_counts > 0])
    print("\nColumns with null values:\n", null_counts[null_counts > 0])
    print("\nFinal shape:", df.shape)

    return df
    


In [28]:
df_test=pd.read_csv(datapath+"df_test_str.csv",low_memory=False)
df_test = preprocess_test_df(df_test)
df_test.to_csv(datapath+"processed_test_set.csv", index=False)

del df_test
gc.collect()


C:\Users\loke\AppData\Local\Temp\ipykernel_1596\3597887986.py:15: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')


Numeric columns with negative values:
 Init Bwd Win Byts    1003202
Init Fwd Win Byts     546364
dtype: int64

Numeric columns with infinite values:
 Series([], dtype: int64)

Columns with null values:
 Series([], dtype: int64)

Final shape: (1952765, 77)


0

### Encode Label and  Attack_type 

In [29]:
import pandas as pd
import json

df_train=pd.read_csv(datapath+"processed_train_set.csv",low_memory=False)
df_val=pd.read_csv(datapath+"processed_validation_set.csv",low_memory=False)
df_test=pd.read_csv(datapath+"processed_test_set.csv",low_memory=False)

# ----------------- Load mapping.json -----------------
with open(r"D:\fypadpm\proj1\mappings.json", "r") as f:
    mapping_data = json.load(f)

# Mappings from JSON (int → string)
label_mapping = mapping_data["label_mapping"]
attack_type_mapping = mapping_data["attack_type_mapping"]

# ----------------- Reverse mappings (string → int) -----------------
label_reverse_mapping = {v: int(k) for k, v in label_mapping.items()}
attack_type_reverse_mapping = {v: int(k) for k, v in attack_type_mapping.items()}

# ----------------- Apply mapping to df -----------------
# df must already be loaded before this

for df in [df_train, df_val, df_test]:
    df['Label'] = df['Label'].map(label_reverse_mapping).astype(int)
    df['Attack_Type'] = df['Attack_Type'].map(attack_type_reverse_mapping).astype(int)

# # ----------------- Save mappings (optional) -----------------
# joblib.dump(label_reverse_mapping, "D:/r6g/prog1/cicids2018M3/label_reverse_mapping.pkl")
# joblib.dump(attack_type_reverse_mapping, "D:/r6g/prog1/cicids2018M3/attack_type_reverse_mapping.pkl")

print("Label reverse mapping:", label_reverse_mapping)
print("Attack Type reverse mapping:", attack_type_reverse_mapping)

df_train.to_csv(datapath+'encoded_train_set.csv', index=False)
df_val.to_csv(datapath+'encoded_validation_set.csv', index=False)
df_test.to_csv(datapath+'encoded_test_set.csv', index=False)


del df_train, df_val, df_test
gc.collect()


Label reverse mapping: {'Benign': 0, 'Bot': 1, 'Brute Force -Web': 2, 'Brute Force -XSS': 3, 'DDOS attack-HOIC': 4, 'DDOS attack-LOIC-UDP': 5, 'DDoS attacks-LOIC-HTTP': 6, 'DoS attacks-GoldenEye': 7, 'DoS attacks-Hulk': 8, 'DoS attacks-SlowHTTPTest': 9, 'DoS attacks-Slowloris': 10, 'FTP-BruteForce': 11, 'Infilteration': 12, 'SQL Injection': 13, 'SSH-Bruteforce': 14}
Attack Type reverse mapping: {'Benign': 0, 'Bot': 1, 'Brute Force': 2, 'DDoS': 3, 'DoS': 4, 'Infiltration': 5, 'Web Attack': 6}


0

### Normalize Numerical Feature

In [30]:
from sklearn.preprocessing import MinMaxScaler
import joblib
import pandas as pd

# Load train set
df_train = pd.read_csv(datapath+"encoded_train_set.csv",low_memory=False)

exclude_cols = ['Attack_Type', 'Label']

# 1️⃣ Choose numeric feature columns ONLY (exclude targets)
feature_numeric_cols = (
    df_train.drop(columns=exclude_cols, errors='ignore')
            .select_dtypes(include=['number'])
            .columns.tolist()
)

# 2️⃣ Save numeric columns for future use
joblib.dump(feature_numeric_cols, datapath+"feature_numeric_cols.pkl")

# 3️⃣ Scale numeric features in-place
scaler = MinMaxScaler()
df_train[feature_numeric_cols] = scaler.fit_transform(df_train[feature_numeric_cols])

# 4️⃣ Save the fitted scaler
joblib.dump(scaler, datapath+"minmax_scaler_on_feature_numeric_cols.pkl")

# 5️⃣ Save the normalized train set
df_train.to_csv(datapath+'normalized_train_set.csv', index=False)

# ==============================
# Scale validation and test sets
# ==============================

df_val = pd.read_csv(datapath+"encoded_validation_set.csv",low_memory=False)
df_val[feature_numeric_cols] = scaler.transform(df_val[feature_numeric_cols])
df_val[feature_numeric_cols] = df_val[feature_numeric_cols].clip(lower=0, upper=1)
df_val.to_csv(datapath+'normalized_validation_set.csv', index=False)

df_test = pd.read_csv(datapath+"encoded_test_set.csv",low_memory=False)
df_test[feature_numeric_cols] = scaler.transform(df_test[feature_numeric_cols])
df_test[feature_numeric_cols] = df_test[feature_numeric_cols].clip(lower=0, upper=1)
df_test.to_csv(datapath+'normalized_test_set.csv', index=False)

print("Normalization completed and saved successfully.")


del df_train, df_val, df_test

Normalization completed and saved successfully.


<!-- ### Split train and validation Dataset Stratically -->

### Feature Selection

#### Feature Selection: Boruta

In [31]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib

# --------------------------
# Load train dataset
# --------------------------
df_train = pd.read_csv(datapath+"normalized_train_set.csv",low_memory=False)
X = df_train.drop(['Attack_Type', 'Label'], axis=1).select_dtypes(include=[np.number])
y = df_train['Attack_Type']

# --------------------------
# Boruta feature selection
# --------------------------
rf = RandomForestClassifier(
    n_jobs=-1,
    max_depth=5,
    n_estimators=50,
    random_state=42
)
boruta = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    max_iter=50,
    random_state=42
)
boruta.fit(X.values, y.values)

# Selected features
selected_features = X.columns[boruta.support_].tolist()
print("Boruta selected features:", selected_features)

# Feature importance
orig_importances = boruta.estimator.feature_importances_[:len(X.columns)]
ranking_df = pd.DataFrame({
    'feature': X.columns,
    'importance': orig_importances,
    'selected_by_boruta': boruta.support_
}).sort_values(by='importance', ascending=False)


top_10_features = ranking_df.head(10)['feature'].tolist()
top_20_features = ranking_df.head(20)['feature'].tolist()
top_30_features = ranking_df.head(30)['feature'].tolist()

print("Top 10:", top_10_features)
print("Top 20:", top_20_features)
print("Top 30:", top_30_features)

# --------------------------
# Save all top sets in ONE PKL dictionary
# --------------------------
all_feature_sets = {
    "top10": top_10_features,
    "top20": top_20_features,
    "top30": top_30_features
}

joblib.dump(all_feature_sets, datapath + "boruta_top_feature_sets.pkl")
print("Saved PKL: boruta_top_feature_sets.pkl")

# --------------------------
# Build final datasets
# --------------------------
def save_boruta_dataset(df, features, out_path):
    df_final = df[features + ['Label', 'Attack_Type']]
    df_final.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df_final


df_val = pd.read_csv(datapath+"normalized_validation_set.csv",low_memory=False)
df_test = pd.read_csv(datapath+"normalized_test_set.csv",low_memory=False)

save_boruta_dataset(df_train, top_10_features, datapath + "Boruta_train_t10.csv")
save_boruta_dataset(df_val,   top_10_features, datapath + "Boruta_validation_t10.csv")
save_boruta_dataset(df_test,  top_10_features, datapath + "Boruta_test_t10.csv")

# -------- Top 20 --------
save_boruta_dataset(df_train, top_20_features, datapath + "Boruta_train_t20.csv")
save_boruta_dataset(df_val,   top_20_features, datapath + "Boruta_validation_t20.csv")
save_boruta_dataset(df_test,  top_20_features, datapath + "Boruta_test_t20.csv")

# -------- Top 30 --------
save_boruta_dataset(df_train, top_30_features, datapath + "Boruta_train_t30.csv")
save_boruta_dataset(df_val,   top_30_features, datapath + "Boruta_validation_t30.csv")
save_boruta_dataset(df_test,  top_30_features, datapath + "Boruta_test_t30.csv")

print("All Boruta top-N datasets created successfully.")


Boruta selected features: ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Ini

#### Feature Selection using CorrMI

In [32]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_selection import mutual_info_classif

# --- Load datasets ---
df_train = pd.read_csv(datapath+"normalized_train_set.csv",low_memory=False)

# --- Features & target ---
X = df_train.drop(['Attack_Type', 'Label'], axis=1).select_dtypes(include=[np.number])
y = df_train['Attack_Type']

# --- Correlation Filtering ---
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]
X_filtered = X.drop(columns=to_drop)

# --- Mutual Information ---
mi = mutual_info_classif(X_filtered, y, random_state=42)
mi_df = pd.DataFrame({'Feature': X_filtered.columns, 'MI_Score': mi})

mi_sorted = mi_df.sort_values(by="MI_Score", ascending=False)

top10_features = mi_sorted.head(10)['Feature'].tolist()
top20_features = mi_sorted.head(20)['Feature'].tolist()
top30_features = mi_sorted.head(30)['Feature'].tolist()

print("Top 10 MI features:", top10_features)
print("Top 20 MI features:", top20_features)
print("Top 30 MI features:", top30_features)

# -----------------------------------------
# 4. Save ALL feature sets in ONE PKL
# -----------------------------------------
all_feature_sets = {
    "top10": top10_features,
    "top20": top20_features,
    "top30": top30_features
}

joblib.dump(all_feature_sets, datapath + "corrmi_feature_sets.pkl")
print("Saved: corrmi_feature_sets.pkl (top10/top20/top30)")

# -----------------------------------------
# 5. Function to Save Filtered Datasets
# -----------------------------------------
def save_corrmi_dataset(df, features, out_path):
    df_final = df[features + ['Label', 'Attack_Type']]
    df_final.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df_final

# -----------------------------------------
# 6. Create Train/Val/Test Sets for Each Top-N
# -----------------------------------------

# ---- Train ----
save_corrmi_dataset(df_train, top10_features, datapath + "CorrMI_train_t10.csv")
save_corrmi_dataset(df_train, top20_features, datapath + "CorrMI_train_t20.csv")
save_corrmi_dataset(df_train, top30_features, datapath + "CorrMI_train_t30.csv")

# ---- Validation ----
df_val = pd.read_csv(datapath+"normalized_validation_set.csv",low_memory=False)

save_corrmi_dataset(df_val,   top10_features, datapath + "CorrMI_validation_t10.csv")
save_corrmi_dataset(df_val,   top20_features, datapath + "CorrMI_validation_t20.csv")
save_corrmi_dataset(df_val,   top30_features, datapath + "CorrMI_validation_t30.csv")

# ---- Test ----
df_test = pd.read_csv(datapath+"normalized_test_set.csv",low_memory=False)

save_corrmi_dataset(df_test,  top10_features, datapath + "CorrMI_test_t10.csv")
save_corrmi_dataset(df_test,  top20_features, datapath + "CorrMI_test_t20.csv")
save_corrmi_dataset(df_test,  top30_features, datapath + "CorrMI_test_t30.csv")

print("All Corr+MI datasets generated successfully.")


Top 10 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Seg Size Min', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Protocol', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max']
Top 20 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Seg Size Min', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Protocol', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Duration', 'Flow Pkts/s', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Tot Bwd Pkts', 'Flow Byts/s', 'Tot Fwd Pkts', 'Flow IAT Std', 'Bwd IAT Tot']
Top 30 MI features: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Seg Size Min', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Protocol', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Duration', 'Flow Pkts/s', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Tot Bwd Pkts', 'Flow Byts/s', 'Tot Fwd Pkts', 'Flow IAT Std', 'Bwd IAT Tot', 'Down/Up Ratio', 'Bwd IAT Mean', 'Bwd IAT Std', 'ACK Flag Cnt', 'RST Flag Cnt', 'Src Port', 'F

In [36]:
import pandas as pd

datapath = "D:/fypadpm/dataset/CSECICIDS2018/"

df_check = pd.read_csv(datapath+"normalized_train_set.csv", low_memory=False, nrows=5)
print(f"列数：{df_check.shape[1]}")
print(f"行数检查...")

import os
size = os.path.getsize(datapath+"normalized_train_set.csv") / (1024*1024)
print(f"文件大小：{size:.0f} MB")

列数：77
行数检查...
文件大小：5685 MB


## Dataset Preparation for Mitigation

In [1]:
# --------------------------------------------------------------------------------------------------------
# Preprocessing:
#    1) From processed train_val & test datasets perform one-hot on Attack_Type to generate M1, M2,... M16
#    2) Stratify split train and validation datasets
#    3) Normalize Features
#    4) Feature selection for multi-label best 20 for all labels or union of best 20 from all labels
# -----------------------------------------------------------------------------------------------------------

# datapath
datapath = 'D:/r6g/prog1/cicids2018M3/setc/'

### Step 1 - One-hot Encoding on Attack Types

In [33]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import json
import gc


import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

def encode_label_n_attack_type(df, label_mapping, attack_type_mapping):
    """
    Encode 'Label' and 'Attack_Type' in a dataframe using mappings.json.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing string Label and Attack_Type.
    
    mapping_json_path : str
        Full path to mappings.json
    
    Returns
    -------
    pandas.DataFrame
        DataFrame with encoded Label and Attack_Type (int)
    """
    

    # Reverse mappings (string → int)
    label_reverse_mapping = {v: int(k) for k, v in label_mapping.items()}
    attack_type_reverse_mapping = {v: int(k) for k, v in attack_type_mapping.items()}

    # Encode (map strings → ints)
    df["Label"] = df["Label"].map(label_reverse_mapping).astype(int)
    df["Attack_Type"] = df["Attack_Type"].map(attack_type_reverse_mapping).astype(int)

    return df



def one_hot_encode_attack_type(df_corr, attack_type_mitigation_mapping):
    """
    Convert Attack_Type in df_corr to multilabel one-hot encoding
    based on attack_type_mitigation_mapping.

    Parameters:
    - df_corr: pd.DataFrame containing 'Attack_Type' column
    - attack_type_mitigation_mapping: dict mapping attack types to mitigation labels
    
    Returns:
    - df_final: original df_corr with one-hot encoded mitigation columns added
    """
    X = df_corr.drop(columns=['Attack_Type'])
    y = df_corr[['Attack_Type']]
    
    # Convert attack types to multilabel format
    y_labels = y['Attack_Type'].map(lambda x: attack_type_mitigation_mapping.get(x, []))
    
    ordered_labels = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16']
    
    # One-hot encode attack types
    mlb = MultiLabelBinarizer(classes=ordered_labels)
    y_transformed = mlb.fit_transform(y_labels)
    
    y_encoded_df = pd.DataFrame(y_transformed, columns=ordered_labels)
    df_final = pd.concat([X,y, y_encoded_df], axis=1)
    
    return df_final

In [35]:
# Load the mapping file
with open(r"D:\fypadpm\proj1\mappings.json", "r") as f:
    mapping_data = json.load(f)


# Extract mappings from JSON (int → string)
label_mapping = mapping_data["label_mapping"]
attack_type_mapping = mapping_data["attack_type_mapping"]

# Extract the attack_type_mitigation_mapping
attack_type_mitigation_mapping = mapping_data.get("attack_type_mitigation_mapping", {})
attack_type_mitigation_mapping_int = {int(k): v for k, v in attack_type_mitigation_mapping.items()}
# Inspect the mapping
print(attack_type_mitigation_mapping_int)

# Load dataset
df = pd.concat([
    pd.read_csv(datapath+"df_train_com.csv", low_memory=False),
    pd.read_csv(datapath+"df_validation_com.csv", low_memory=False),
    pd.read_csv(datapath+"df_validation_com.csv", low_memory=False)
], axis=0, ignore_index=True)


# encode Label and Attack_Type
df = encode_label_n_attack_type(df, label_mapping, attack_type_mapping)

# one-hot
df = one_hot_encode_attack_type(df, attack_type_mitigation_mapping_int)

df.to_csv(datapath+'encoded_mitigation_data1.csv', index=False)

del df
gc.collect()

{1: ['M1', 'M2', 'M3', 'M8', 'M9', 'M10', 'M14', 'M15', 'M16'], 2: ['M2', 'M4', 'M5', 'M6', 'M7', 'M13', 'M14'], 3: ['M1', 'M3', 'M4', 'M5', 'M7', 'M8', 'M15', 'M16'], 4: ['M1', 'M3', 'M4', 'M5', 'M7', 'M8', 'M15', 'M16'], 5: ['M2', 'M4', 'M5', 'M6', 'M7', 'M13', 'M14'], 6: ['M1', 'M2', 'M6', 'M7', 'M9', 'M15', 'M16']}


FileNotFoundError: [Errno 2] No such file or directory: 'D:/fypadpm/dataset/CSECICIDS2018/df_train_com.csv'

### Step 2 - Stratify split train & validation datasets

In [4]:
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import gc

# -----------------------------
# Load datasets
# -----------------------------
df = pd.read_csv(datapath+"encoded_mitigation_data1.csv",low_memory=False)

# -----------------------------
# Define multilabel columns
# -----------------------------
ml_labels = ['M1','M2','M3','M4','M5','M6','M7','M8','M9', 'M10','M11','M12','M13','M14','M15','M16']

# -----------------------------
# Features (including Label and Attack_Type)
# -----------------------------
feature_cols = [c for c in df.columns if c not in ml_labels]
X = df[feature_cols]
y = df[ml_labels]

# -----------------------------
# Multilabel Stratified Split
# -----------------------------
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

for train_idx, tmp_idx in msss.split(X, y):
    X_train, X_tmp = X.iloc[train_idx], X.iloc[tmp_idx]
    y_train, y_tmp = y.iloc[train_idx], y.iloc[tmp_idx]

X = X_tmp
y = y_tmp

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for val_idx, test_idx in msss.split(X, y):
    X_val, X_test = X.iloc[val_idx], X.iloc[test_idx]
    y_val, y_test = y.iloc[val_idx], y.iloc[test_idx]


# -----------------------------
# Combine features + multilabels
# -----------------------------
df_train_split = pd.concat([X_train, y_train], axis=1)
df_val_split   = pd.concat([X_val, y_val], axis=1)
df_test_split  = pd.concat([X_test, y_test], axis=1)
# -----------------------------
# Save final datasets
# -----------------------------
df_train_split.to_csv(datapath+'encoded_mitigation_train_set.csv', index=False)
df_val_split.to_csv(datapath+'encoded_mitigation_validation_set.csv', index=False)
df_test_split.to_csv(datapath+'encoded_mitigation_test_set.csv', index=False)

print("Final split complete.")
print("Train shape:", df_train_split.shape)
print("Validation shape:", df_val_split.shape)
print("Test shape:", df_test_split.shape)

del df, df_train_split, df_val_split, df_test_split, X, y, X_tmp, y_tmp
gc.collect()

Final split complete.
Train shape: (11092835, 101)
Validation shape: (2377036, 101)
Test shape: (2377037, 101)


0

### Step 3 - Data Pre-processing

In [5]:
import pandas as pd
import numpy as np

def preprocess_mitigation_train_val_df(df):

    # 1) Drop Date
    if "Date" in df.columns:
        df.drop(columns="Date", inplace=True)

    # 2) Timestamp conversion
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")

    # 3) Numeric conversion
    ml_labels = ['M1','M2','M3','M4','M5','M6','M7','M8','M9','M10','M11','M12','M13','M14','M15','M16']

    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID'] + ml_labels
    cols_to_convert = [col for col in df.columns if col not in exclude_cols]

    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Convert ML labels if needed
    for m in ml_labels:
        if m in df.columns:
            df[m] = pd.to_numeric(df[m], errors='coerce')

    # 4) Remove negative Flow IAT Min/Max (safe in-place)
    if "Flow IAT Min" in df.columns:
        df.drop(df[df["Flow IAT Min"] < 0].index, inplace=True)
    if "Flow IAT Max" in df.columns:
        df.drop(df[df["Flow IAT Max"] < 0].index, inplace=True)

    # 5) Remove inf rows
    cols_inf = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if cols_inf:
        df.drop(df[df[cols_inf].isin([np.inf, -np.inf]).any(axis=1)].index, inplace=True)

    # 6) Fill string columns
    for col in ["Dst IP", "Src IP", "Flow ID"]:
        if col in df.columns:
            df[col] = df[col].fillna("<<M>>")

    # 7) Fill Src Port
    if "Src Port" in df.columns:
        df["Src Port"] = df["Src Port"].fillna(0)

    # 8) Drop NaN in critical columns
    critical = [c for c in ["Flow Byts/s", "Flow Pkts/s"] if c in df.columns]
    if critical:
        df.dropna(subset=critical, inplace=True)

    # 9) Drop all-zero numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols)
    zero_cols = numeric_cols[(df[numeric_cols] == 0).all()]
    if len(zero_cols) > 0:
        df.drop(columns=list(zero_cols), inplace=True)

    # 10) Diagnostics
    numeric_cols = df.select_dtypes(include=['number']).columns.difference(exclude_cols)

    neg_cols = (df[numeric_cols] < 0).sum()
    inf_cols = np.isinf(df[numeric_cols]).sum()
    null_cols = df.isnull().sum()

    print("Columns with negative values:")
    print(neg_cols[neg_cols > 0])

    print("\nColumns with inf values:")
    print(inf_cols[inf_cols > 0])

    print("\nColumns with null values:")
    print(null_cols[null_cols > 0])

    print("\nFinal shape:", df.shape)

    return df



In [6]:
df_train = pd.read_csv(datapath+"encoded_mitigation_train_set.csv", low_memory=False)
df_train = preprocess_mitigation_train_val_df(df_train)
df_val = pd.read_csv(datapath+"encoded_mitigation_validation_set.csv", low_memory=False)
df_val = preprocess_mitigation_train_val_df(df_val)

df_train.to_csv(datapath+"processed_mitigation_train.csv", index=False)
df_val.to_csv(datapath+"processed_mitigation_validation.csv", index=False)

print("train shape:",df_train.shape)
print("val shape:", df_val.shape)

del df_train,df_val
gc.collect()


C:\Users\henry\AppData\Local\Temp\ipykernel_82168\1034799441.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")


Columns with negative values:
Init Bwd Win Byts    5672589
Init Fwd Win Byts    3107678
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (11026797, 97)


C:\Users\henry\AppData\Local\Temp\ipykernel_82168\1034799441.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")


Columns with negative values:
Init Bwd Win Byts    1214566
Init Fwd Win Byts     664652
dtype: int64

Columns with inf values:
Series([], dtype: int64)

Columns with null values:
Series([], dtype: int64)

Final shape: (2362861, 97)
train shape: (11026797, 97)
val shape: (2362861, 97)


0

In [7]:
import pandas as pd
import numpy as np

def preprocess_mitigation_test_df(df):
    """
    Preprocess a single DataFrame according to the following steps:
    - Drop unnecessary columns
    - Convert timestamp
    - Convert numeric columns, handle NaN/inf/-inf
    - Fill missing string/numeric columns
    - Remove negative values (except allowed columns)
    - Drop all-zero numeric columns
    - Final validation
    """
    
    # -------------------------
    # Drop Date column
    # -------------------------
    if 'Date' in df.columns:
        df = df.drop(columns=['Date'])
    
    # -------------------------
    # Convert Timestamp (fixed warning)
    # -------------------------
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(
            df['Timestamp'],
            format="%Y-%m-%d %H:%M:%S",
            errors='coerce'
        )
    
    # -------------------------
    # Convert columns to numeric
    # -------------------------
    ml_labels = [f"M{i}" for i in range(1, 17)]
    exclude_cols = ['Timestamp', 'Label', 'Attack_Type', 'Src IP', 'Dst IP', 'Flow ID'] + ml_labels

    cols_to_convert = [col for col in df.columns if col not in exclude_cols]
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # -------------------------
    # Fill missing string columns
    # -------------------------
    for col in ['Dst IP', 'Src IP', 'Flow ID']:
        if col in df.columns:
            df[col] = df[col].fillna("<<M>>")
    
    # -------------------------
    # Fill missing numeric
    # -------------------------
    for col in ['Src Port', 'Dst Port', 'Protocol']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # -------------------------
    # Drop NaN in critical numeric columns
    # -------------------------
    critical_numeric = ['Flow Byts/s', 'Flow Pkts/s']
    df = df.dropna(subset=[c for c in critical_numeric if c in df.columns])
    
    # -------------------------
    # Remove inf/-inf (safe, no SettingWithCopyWarning)
    # -------------------------
    numeric_cols = (
        df.select_dtypes(include=['number'])
          .columns
          .difference(exclude_cols)
          .tolist()
    )

    df.loc[:, numeric_cols] = df.loc[:, numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    
    # -------------------------
    # Remove negative values (except allowed)
    # -------------------------
    allowed_negative_cols = ['Init Fwd Win Byts', 'Init Bwd Win Byts']

    cols_to_check_neg = [col for col in numeric_cols if col not in allowed_negative_cols]

    if cols_to_check_neg:
        df = df[(df[cols_to_check_neg] >= 0).all(axis=1)]
    
    # -------------------------
    # Drop all-zero numeric columns (fixed syntax)
    # -------------------------
    numeric_cols = (
        df.select_dtypes(include=['number'])
          .columns
          .difference(exclude_cols)
          .tolist()
    )

    zero_cols = [col for col in numeric_cols if (df[col] == 0).all()]

    if zero_cols:
        df = df.drop(columns=zero_cols)
    
    # -------------------------
    # Final validation
    # -------------------------
    numeric_cols = (
        df.select_dtypes(include=['number'])
          .columns
          .difference(exclude_cols)
          .tolist()
    )

    neg_counts = (df[numeric_cols] < 0).sum()
    inf_counts = np.isinf(df[numeric_cols]).sum()
    null_counts = df.isna().sum()
    
    print("Numeric columns with negative values:\n", neg_counts[neg_counts > 0])
    print("\nNumeric columns with infinite values:\n", inf_counts[inf_counts > 0])
    print("\nColumns with null values:\n", null_counts[null_counts > 0])
    print("\nFinal shape:", df.shape)
    
    return df


In [8]:
df_test=pd.read_csv(datapath+"encoded_mitigation_test_set.csv",low_memory=False)
# print(df_test.columns)
df_test = preprocess_mitigation_test_df(df_test)
# print(df_test.columns)
df_test.to_csv(datapath+"processed_mitigation_test.csv", index=False)

del df_test
gc.collect()


Numeric columns with negative values:
 Init Bwd Win Byts    1216099
Init Fwd Win Byts     666226
dtype: int64

Numeric columns with infinite values:
 Series([], dtype: int64)

Columns with null values:
 Series([], dtype: int64)

Final shape: (2362671, 97)


0

### Step 4 - Normalizing train, val and test datasets

In [9]:
from sklearn.preprocessing import MinMaxScaler
import joblib
import pandas as pd
import gc

# Load train set
df_train = pd.read_csv(datapath+"processed_mitigation_train.csv",low_memory=False)

ml_labels = ['M1','M2','M3','M4','M5','M6','M7','M8','M9', 'M10','M11','M12','M13','M14','M15','M16']

exclude_cols = ['Attack_Type', 'Label'] + ml_labels

# 1️⃣ Choose numeric feature columns ONLY (exclude targets)
feature_numeric_cols = (
    df_train.drop(columns=exclude_cols, errors='ignore')
            .select_dtypes(include=['number'])
            .columns.tolist()
)

# 2️⃣ Save numeric columns for future use
joblib.dump(feature_numeric_cols, datapath+"mitigation_feature_numeric_cols.pkl")

# 3️⃣ Scale numeric features in-place
scaler = MinMaxScaler()
df_train[feature_numeric_cols] = scaler.fit_transform(df_train[feature_numeric_cols])

# 4️⃣ Save the fitted scaler
joblib.dump(scaler, datapath+"minmax_scaler_on_mitigation_feature_numeric_cols.pkl")

# 5️⃣ Save the normalized train set
df_train.to_csv(datapath+'normalized_mitigation_train.csv', index=False)

# ==============================
# Scale validation and test sets
# ==============================

df_val = pd.read_csv(datapath+"processed_Mitigation_validation.csv",low_memory=False)
df_val[feature_numeric_cols] = scaler.transform(df_val[feature_numeric_cols])
df_val[feature_numeric_cols] = df_val[feature_numeric_cols].clip(lower=0, upper=1)
df_val.to_csv(datapath+'normalized_mitigation_validation.csv', index=False)

df_test = pd.read_csv(datapath+"processed_mitigation_test.csv",low_memory=False)
df_test[feature_numeric_cols] = scaler.transform(df_test[feature_numeric_cols])
df_test[feature_numeric_cols] = df_test[feature_numeric_cols].clip(lower=0, upper=1)
df_test.to_csv(datapath+'normalized_mitigation_test.csv', index=False)

print("Normalization completed and saved successfully.")


del df_train, df_val, df_test
gc.collect()

Normalization completed and saved successfully.


0

### Step 5 - Multi-Label Feature Selections

#### Multi-Label Feature Selection using CorrMI

In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_selection import mutual_info_classif
from joblib import Parallel, delayed
import gc

# ============================
# Load datasets
# ============================
df_train = pd.read_csv(datapath + "normalized_mitigation_train.csv", low_memory=False)

# ============================
# Define feature columns & labels
# ============================
LABELS = [f"M{i}" for i in range(1, 17)]

# Drop non-numeric safely
ALL_NUMERIC_FEATURES = df_train.drop(['Attack_Type', 'Label'] + LABELS, axis=1, errors='ignore')\
                               .select_dtypes(include=[np.number]).columns.tolist()

# ============================
# Correlation filtering
# ============================
corr_matrix = df_train[ALL_NUMERIC_FEATURES].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

print(f"Correlation filtered: {len(high_corr_drop)} features removed")
X_filtered = df_train[ALL_NUMERIC_FEATURES].drop(columns=high_corr_drop)
X_filtered = X_filtered.fillna(0).replace([np.inf, -np.inf], 0)
filtered_feature_list = X_filtered.columns.tolist()

# ============================
# Mutual Information computation (parallel)
# ============================

def compute_mi(label):
    print(f"Running CorrMI for {label} ...")
    y = df_train[label]
    mi_scores = mutual_info_classif(X_filtered, y, random_state=42)
    mi_df = pd.DataFrame({'Feature': filtered_feature_list, 'MI': mi_scores})
    mi_df = mi_df.sort_values(by='MI', ascending=False)

    top10 = mi_df.head(10)['Feature'].tolist()
    top20 = mi_df.head(20)['Feature'].tolist()
    top30 = mi_df.head(30)['Feature'].tolist()

    return label, {'top10': top10, 'top20': top20, 'top30': top30}, top30

# Use all CPU cores
results = Parallel(n_jobs=8)(delayed(compute_mi)(lbl) for lbl in LABELS)

# Collect results
top_features_per_label = {}
union_top30_features = set()

for label, top_dict, top30 in results:
    top_features_per_label[label] = top_dict
    union_top30_features.update(top30)
    print(f"\nLabel: {label} | Top10: {top_dict['top10']}")
    print(f"Top20: {top_dict['top20']}")
    print(f"Top30: {top_dict['top30']}")

# ============================
# Save feature info
# ============================
all_feature_info = {
    'per_label_top_features': top_features_per_label,
    'union_top30_features': list(union_top30_features)
}

joblib.dump(all_feature_info, datapath + "mitigation_corrmi_top_features.pkl")
print(f"\nSaved all features to: mitigation_corrmi_top_features.pkl")
print(f"Union of top30 features count: {len(union_top30_features)}")

# ============================
# Save datasets using union feature set
# ============================
def save_corrmi_dataset(df, feature_list, out_path):
    existing_features = [c for c in feature_list if c in df.columns]
    df_out = df[existing_features + LABELS + ['Label', 'Attack_Type']]
    df_out.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | Shape: {df_out.shape}")
    return df_out

df_train_final = save_corrmi_dataset(df_train, union_top30_features,
                                     datapath + "CorrMI_mitigation_train.csv")

df_val   = pd.read_csv(datapath + "normalized_mitigation_validation.csv", low_memory=False)

save_corrmi_dataset(df_val, union_top30_features,
                                     datapath + "CorrMI_mitigation_validation.csv")

df_test  = pd.read_csv(datapath + "normalized_mitigation_test.csv", low_memory=False)
save_corrmi_dataset(df_test, union_top30_features,
                                     datapath + "CorrMI_mitigation_test.csv")

print("\nPreview train dataset:")
print(df_train_final.head())

del df_train, df_val, df_test, df_train_final
gc.collect()


Correlation filtered: 37 features removed

Label: M1 | Top10: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Pkt Len Max', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s']
Top20: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Pkt Len Max', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s', 'Tot Bwd Pkts', 'Flow Duration', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Tot Fwd Pkts', 'Flow Byts/s', 'ACK Flag Cnt', 'Protocol', 'Flow IAT Std', 'Bwd IAT Tot']
Top30: ['Init Fwd Win Byts', 'Dst Port', 'Fwd Pkt Len Max', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'TotLen Fwd Pkts', 'Pkt Len Var', 'Flow IAT Max', 'Flow IAT Mean', 'Flow Pkts/s', 'Tot Bwd Pkts', 'Flow Duration', 'Bwd Pkts/s', 'Init Bwd Win Byts', 'Tot Fwd Pkts', 'Flow Byts/s', 'ACK Flag Cnt', 'Protocol', 'Flow IAT Std', 'Bwd IAT Tot', 'Bwd IAT Std', 'Bwd IAT Mean', 'Fwd Seg Size Min', 'Fwd Pkt Len Min', 'Bwd Pkt Len Min

0

#### Multi-Label Feature Selectin using Boruta

In [ ]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import joblib
import gc

# ============================
# Load train dataset
# ============================
df_train = pd.read_csv(datapath + "normalized_mitigation_train.csv", low_memory=False)

# ============================
# Labels
# ============================
LABELS = [f"M{i}" for i in range(1, 17)]

# ============================
# Numeric features
# ============================
X_all = df_train.drop(['Attack_Type', 'Label'] + LABELS, axis=1, errors='ignore').select_dtypes(include=[np.number])

# Clean data for Boruta
X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0)
feature_cols = X_all.columns.tolist()
print(f"feature_cols {feature_cols}")
# ============================
# Run Boruta per label
# ============================
top_features_per_label = {}
union_top30_features = set()

for lbl in LABELS:
    print(f"\n========== Running Boruta for {lbl} ==========")

    y = df_train[lbl].values
    # X = X_all.values
    X = X_all          # ← Keep as DataFrame


    # - try to speed up--------
    rf = RandomForestClassifier(
        n_jobs=-1,
        max_depth=8,
        n_estimators=50,    # <<< major speedup
        random_state=42
    )
    
    boruta = BorutaPy(
        estimator=rf,
        n_estimators=50,    # <<< overrides auto (faster)
        max_iter=20,        # <<< limits iterations
        random_state=42,
        verbose=0
    )
    # ---------------------------------

    # # Base estimator
    # rf = RandomForestClassifier(
    #     n_jobs=-1, 
    #     max_depth=5, 
    #     random_state=42
    # )

    # boruta = BorutaPy(
    #     estimator=rf,
    #     n_estimators='auto',
    #     random_state=42,
    #     verbose=0
    # )

    boruta.fit(X, y)
    print("After Fit")
    
    # # Correct arrays
    # selected_mask = boruta.support_                      # boolean mask
    # importances = boruta.estimator_.feature_importances_ # correct length
    # # Ranking dataframe
    # ranking_df = pd.DataFrame({
    #     'feature': feature_cols,
    #     'importance': importances,
    #     'selected': selected_mask
    # }).sort_values(by='importance', ascending=False)


    selected_mask = boruta.support_
    orig_importances = boruta.estimator.feature_importances_[:len(feature_cols)]
    
    ranking_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': orig_importances,
        'selected_by_boruta': selected_mask
    }).sort_values(by='importance', ascending=False)

               
    # orig_importances = boruta.estimator.feature_importances_[:len(X.columns)]
    # ranking_df = pd.DataFrame({
    #     'feature': X.columns,
    #     'importance': orig_importances,
    #     'selected_by_boruta': boruta.support_
    # }).sort_values(by='importance', ascending=False)

    # Extract top features
    top10 = ranking_df.head(10)['feature'].tolist()
    top20 = ranking_df.head(20)['feature'].tolist()
    top30 = ranking_df.head(30)['feature'].tolist()

    top_features_per_label[lbl] = {
        'top10': top10,
        'top20': top20,
        'top30': top30
    }

    union_top30_features.update(top30)

    print(f"Boruta selected ({lbl}): {selected_mask.sum()} features")
    print(f"Top10: {top10}")
    print(f"Top20: {top20}")
    print(f"Top30: {top30}")

# ============================
# Save PKL
# ============================
all_feature_info = {
    'per_label_top_features': top_features_per_label,
    'union_top30_features': list(union_top30_features)
}

joblib.dump(all_feature_info, datapath + "mitigation_boruta_top_features.pkl")
print(f"\nSaved PKL: mitigation_boruta_top_features.pkl")
print(f"Union top30 features count: {len(union_top30_features)}")

# ============================
# Save datasets
# ============================
def save_boruta_dataset(df, feature_set, out_path):
    existing = [c for c in feature_set if c in df.columns]
    df_out = df[existing + LABELS + ['Label', 'Attack_Type']]
    df_out.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | Shape: {df_out.shape}")
    return df_out

df_train_final = save_boruta_dataset(df_train, union_top30_features,
                                     datapath + "Boruta_mitigation_train.csv")

df_val   = pd.read_csv(datapath + "normalized_mitigation_validation.csv", low_memory=False)
df_test  = pd.read_csv(datapath + "normalized_mitigation_test.csv", low_memory=False)
save_boruta_dataset(df_val, union_top30_features,
                    datapath + "Boruta_mitigation_validation.csv")

save_boruta_dataset(df_test, union_top30_features,
                    datapath + "Boruta_mitigation_test.csv")

print("\nPreview train:")
print(df_train_final.head())

del df_train, df_val, df_test, df_train_final
gc.collect()


feature_cols ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts', 'Sub